# 15d · 把设计循环做成能重跑的流程

## 这一章在整个项目里的位置

```
第三阶段 G
  ├─ 15 / 15a / 15c / 15b  ✅ 结论都有了
  ├─ 15d 平台化  ← 你在这里
  └─ 15e 讲法
```

## 这一章做什么

前面几章产出的是**结论**。这一章把结论变成**能重复执行的东西**。

一次设计任务要做的事：

```
① 筛查   这个任务是不是在允许的范围内
② 排序   按 15c 的结论，用功能能给位置排序
③ 取预算 最重要的那一批固定住，生成器不许动
④ 生成   ProteinMPNN 写序列
⑤ 审计   检查生成的序列有没有动过固定位点
⑥ 存档   每条设计都带着「谁、用什么版本、固定了哪些」
```

**第 ⑤ 步不是走形式。** 如果固定位点的文件因为名字对不上被静默忽略，
生成出来的序列看着完全正常，而**恰恰改掉了本来要保护的那些位置**。
这个坑我在写这一章时真踩到了——见下面"踩过的坑"。

---

## 设计上的三个决定

### 一、循环本身不碰模型

`qafi_core.design` 里没有一行调用 ProteinMPNN 或我们的模型。它只做"决定哪些位置固定"和"写出
生成器要读的文件"。模型由调用方传进来。

**好处**：这套逻辑可以在没有 GPU 的机器上测，notebook 和 Nextflow 任务跑的是同一份代码。

### 二、筛查是「白名单」不是「黑名单」

```python
ALLOWED_INTENTS = {"stability_engineering", "expression_improvement", "benchmark"}
```

任务必须声明自己是干什么的，**没声明的一律不跑**。
黑名单的问题是漏掉的就放行了；白名单漏掉的是不跑。

### 三、来源记录里有固定集的哈希

两次任务如果固定的位置不同，哈希就不同。**看起来一样的两次任务能被区分开。**

In [ ]:
import os
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
from qafi_core import design

print("允许的任务类型:", sorted(design.ALLOWED_INTENTS))
print("可用的排序依据:", sorted(design.CRITERIA))

In [ ]:
# 范围外的任务，在算任何东西之前就被拒绝
try:
    design.screen("make_a_toxin")
except design.RefusedError as e:
    print("拒绝:", str(e)[:100])

---

## 跑一次真实的任务

拿一个锁死的考试蛋白 O00257（560 个位置），按**功能能**排序，固定一半。

In [ ]:
sites = pd.read_parquet("data/sites/positions.parquet").query("uniprot_id == 'O00257'")
ranked = design.rank_positions(sites[["position", "fold_cost", "func_cost"]], by="func")
fixed = design.fixed_positions(ranked, budget=0.5)

print(f"{len(sites)} 个位置，固定 {len(fixed)} 个")
print("最该保护的前 10 个位置:", fixed[:10] if fixed[:10] == sorted(fixed)[:10] else ranked.position.head(10).tolist())
print("按功能能最贵的前 10 个:", ranked.position.head(10).tolist())

### 整条流程

```bash
cd pipelines/design
nextflow run main.nf -profile local \
    --sites   ../../data/sites/positions.parquet \
    --structures ../../data/structures_test \
    --proteins O00257 --budget 0.5 --n_seq 4 \
    --outdir ../../results/design
```

三个任务依次跑：`PLAN → GENERATE → AUDIT`。

In [ ]:
import json

out = Path("results/design/O00257")
if out.exists():
    audit = json.loads(next(out.glob("*_audit.json")).read_text())
    prov = json.loads(next(out.glob("*_provenance.json")).read_text())
    print(f"设计 {audit['designs']} 条，长度 {audit['length']}，固定 {audit['fixed']} 个位置")
    for r in audit["per_design"]:
        print(f"  设计 {r['design']}: 改了 {r['mutations']} 个位置，违规 {len(r['violations'])} 个")
    print(f"\n全部合规: {audit['clean']}")
    print(f"固定集哈希: {prov['fixed_sha256']}   任务类型: {prov['screening']['intent']}")
else:
    print("先跑上面那条 nextflow 命令")

### 结果怎么看

| | |
|---|---|
| 生成 | 4 条序列，每条改了 **245–259** 个位置 |
| 固定集 | 280 / 560 |
| **动过固定位点的** | **0** |

生成器改了接近一半的位置，**但一个被保护的位置都没碰**。

### 踩过的坑（留下来，因为它正是第 ⑤ 步存在的理由）

**ProteinMPNN 的固定位点文件，键必须是 PDB 文件的名字，不是 UniProt 号。**
对不上的时候它**不报错**，只是当作没有固定位点，把整条序列重写一遍。

我第一版 CLI 就是用 accession 当键的。**如果没有审计这一步，产出看起来完全正常**
——4 条序列、改动数量也合理——而保护完全失效了。

另一个坑：审计一开始是从来源记录里读固定集的**路径**。那在交互式运行时没问题，
但工作流里每个任务在各自的目录里跑，路径必然失效。改成**把文件本身传下去**。

## 没做的部分（要写明）

| | 状态 |
|---|---|
| 本地流程 | ✅ 跑通 |
| **AWS Batch 上的 GPU 队列** | ❌ **没部署**——要 GPU 队列和把 ProteinMPNN 权重打进去的镜像 |
| API 的 `/design` 路由 | ❌ 没做 |
| agent 工具 | ❌ 没做 |

`nextflow.config` 里的 `batch` profile 写的是**部署时会长什么样**，不是跑过的东西。

## 你现在要做什么

想试的话，跑上面那条 nextflow 命令。换 `--criterion fold` 或 `--budget 0.25` 能看到固定集怎么变。

## 记账
代码在 `src/qafi_core/design.py` 和 `design_cli.py`，14 个单元测试在 `tests/unit/test_design.py`。
流程在 `pipelines/design/`。